# Image Studio を Google Colab で動かす

このノートブックは、GPU搭載PCが無くても、Google ColabのGPUを借りて
**ComfyUI** と **Image Studio**(画像生成サイト)を起動し、ブラウザから使えるようにするものです。

## 使い方
1. メニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU** を選択してください(T4などでOK)
2. 上から順番に、セルを1つずつ実行してください(左の再生ボタンを押す、または Shift+Enter)
3. 一番下のセルを実行すると、`https://xxxxx.trycloudflare.com` という**一時的な公開URL**が表示されます
4. そのURLをブラウザで開くと、Image Studioの画面が使えます

## 注意点
- Colab無料版は、**一定時間で接続が切れたり、GPUが割り当てられないことがあります**
- 表示されるURLは実行するたびに変わり、**そのURLを知っている人なら誰でもアクセスできます**。
  SNSなどで共有しないでください
- モデルファイル(数GB)はGoogleドライブに保存するので、2回目以降の起動は少し速くなります


### 1. GPUが使えるか確認

In [ ]:
!nvidia-smi


### 2. Googleドライブをマウント(モデルの保存用)

数GBあるモデルファイルを毎回ダウンロードし直さずに済むよう、Googleドライブに保存します。
初回はGoogleアカウントへのアクセス許可を求められるので、許可してください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
MODELS_ROOT = "/content/drive/MyDrive/image-studio-models"
for sub in ["checkpoints", "unet", "clip", "vae", "ipadapter", "clip_vision"]:
    os.makedirs(f"{MODELS_ROOT}/{sub}", exist_ok=True)
print("モデル保存先:", MODELS_ROOT)


### 3. ComfyUI をインストール

In [ ]:
%cd /content
!test -d ComfyUI || git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt


### 4. モデル置き場をGoogleドライブにつなぐ(symlink)

ComfyUIの `models/` フォルダの中身を、さきほどのGoogleドライブのフォルダに向けます。


In [ ]:
import os

COMFY_MODELS = "/content/ComfyUI/models"
for sub in ["checkpoints", "unet", "clip", "vae", "ipadapter", "clip_vision"]:
    target = f"{COMFY_MODELS}/{sub}"
    source = f"{MODELS_ROOT}/{sub}"
    if os.path.islink(target):
        continue
    if os.path.isdir(target) and not os.path.islink(target):
        # 既存の空フォルダを置き換える
        try:
            os.rmdir(target)
        except OSError:
            pass
    if not os.path.exists(target):
        os.symlink(source, target)
print("モデルフォルダをGoogleドライブにリンクしました")


### 5. SDXLモデルをダウンロード(初回のみ時間がかかります・約6GB)

すでにGoogleドライブにダウンロード済みなら、この処理はスキップされます。


In [ ]:
import os

sdxl_path = f"{MODELS_ROOT}/checkpoints/sd_xl_base_1.0.safetensors"
if os.path.exists(sdxl_path):
    print("すでにダウンロード済みです:", sdxl_path)
else:
    url = "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"
    !wget -q --show-progress -O "{sdxl_path}" "{url}"
    print("ダウンロード完了:", sdxl_path)


### 6. (任意・上級者向け) 参照画像からキャラクターを作る機能(IPAdapter)を有効にする

このセルは**実行しなくても**他の機能(文章から立ち絵・背景を作る機能)は使えます。
「参照画像からキャラクターを作る」機能まで使いたい場合だけ実行してください。
モデルのダウンロードにさらに時間がかかります。


In [ ]:
ENABLE_IPADAPTER = False  # 使いたい場合は True に変更してから実行してください

if ENABLE_IPADAPTER:
    %cd /content/ComfyUI/custom_nodes
    !test -d ComfyUI_IPAdapter_plus || git clone --depth 1 https://github.com/cubiq/ComfyUI_IPAdapter_plus.git
    %cd ComfyUI_IPAdapter_plus
    !pip install -q -r requirements.txt

    import os
    clip_vision_path = f"{MODELS_ROOT}/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"
    if not os.path.exists(clip_vision_path):
        !wget -q --show-progress -O "{clip_vision_path}" "https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors"

    ipadapter_path = f"{MODELS_ROOT}/ipadapter/ip-adapter-plus_sdxl_vit-h.safetensors"
    if not os.path.exists(ipadapter_path):
        !wget -q --show-progress -O "{ipadapter_path}" "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors"

    print("IPAdapterのセットアップが完了しました")
else:
    print("IPAdapterのセットアップをスキップしました(ENABLE_IPADAPTER = True にすると有効化できます)")


### 7. ComfyUI をバックグラウンドで起動

In [ ]:
import subprocess, time, requests

comfyui_log = open("/content/comfyui.log", "w")
comfyui_proc = subprocess.Popen(
    ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=comfyui_log,
    stderr=subprocess.STDOUT,
)

print("ComfyUIの起動を待っています...")
for _ in range(120):
    try:
        requests.get("http://127.0.0.1:8188/", timeout=2)
        print("ComfyUIが起動しました。")
        break
    except Exception:
        time.sleep(3)
else:
    print("ComfyUIの起動確認がタイムアウトしました。/content/comfyui.log を確認してください。")


### 8. Image Studio(このプロジェクト)を取得

In [ ]:
%cd /content
!test -d image-studio && rm -rf image-studio
!git clone --depth 1 https://github.com/jinziruixi07-design/image-studio.git
%cd /content/image-studio
!pip install -q -r requirements.txt


### 9. (任意) AIプロンプト自動変換を使う場合はAPIキーを設定

使わない場合はこのセルをスキップしてください(他の機能はすべて使えます)。

左のサイドバーの鍵アイコン(シークレット)で `ANTHROPIC_API_KEY` という名前で
APIキーを登録しておくと、下のセルが自動で読み込みます。


In [ ]:
import os

try:
    from google.colab import userdata
    key = userdata.get('ANTHROPIC_API_KEY')
    if key:
        os.environ['ANTHROPIC_API_KEY'] = key
        print("ANTHROPIC_API_KEYを設定しました。")
    else:
        print("ANTHROPIC_API_KEYのシークレットが見つかりませんでした。AIプロンプト変換は使えません。")
except Exception as e:
    print("ANTHROPIC_API_KEYは設定されていません。AIプロンプト変換は使えません。", e)


### 10. Image Studio をバックグラウンドで起動

In [ ]:
import subprocess, time, requests, os

env = os.environ.copy()
env["COMFYUI_URL"] = "http://127.0.0.1:8188"
env["HOST"] = "0.0.0.0"

studio_log = open("/content/studio.log", "w")
studio_proc = subprocess.Popen(
    ["python3", "server.py"],
    cwd="/content/image-studio",
    env=env,
    stdout=studio_log,
    stderr=subprocess.STDOUT,
)

print("Image Studioの起動を待っています...")
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:5000/", timeout=2)
        print("Image Studioが起動しました。")
        break
    except Exception:
        time.sleep(2)
else:
    print("Image Studioの起動確認がタイムアウトしました。/content/studio.log を確認してください。")


### 11. 公開URLを発行する(cloudflaredトンネル)

このセルを実行すると、`https://xxxxx.trycloudflare.com` という一時的なURLが表示されます。
それをブラウザで開いてください。**このセルは実行しっぱなしにしておく必要があります**
(止めるとURLが使えなくなります)。


In [ ]:
import subprocess, re, os

%cd /content
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("公開URLを取得しています...\n")
for line in tunnel_proc.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        print("\n\n==================================================")
        print("公開URL:", match.group(0))
        print("==================================================")
        print("このURLをブラウザで開いてください。このセルは実行したままにしてください。")
        break


## 終了のしかた

- 使い終わったら「ランタイム」→「ランタイムを接続解除して削除」を選んでください
- 次回また使うときは、このノートブックを上から順に実行し直してください
  (モデルファイルはGoogleドライブに残っているので、ダウンロードは省略されます)
- URLが動かなくなった場合は、見出し「11. 公開URLを発行する」のセルだけ再実行してみてください
